# 🛍️ Mall Customer Segmentation

**Clustering | CRISP-DM | Executed project | Admin dashboard**

## 1. Business Understanding
Goal: identify actionable customer groups for targeted marketing, loyalty, retention, and campaign design.

## 2. Data Understanding
The popular Mall Customers dataset contains 200 customer records. The notebook downloads the dataset from a public mirror so no CSV must be committed to GitHub.

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

DATA_URL = "https://raw.githubusercontent.com/kennedykwangari/Mall-Customer-Segmentation-Data/master/Mall_Customers.csv"
df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.head())

Dataset shape: (200, 7)
Missing values: 0
Duplicate rows: 0

 CustomerID Gender  Age  Annual Income (k$)  Spending Score (1-100)  Cluster               Segment
          1   Male   19                  15                      39        4      Budget Conscious
          2   Male   21                  15                      81        2 Value-Driven Spenders
          3 Female   20                  16                       6        4      Budget Conscious
          4 Female   23                  16                      77        2 Value-Driven Spenders
          5 Female   31                  17                      40        4      Budget Conscious


## 3. Data Preparation
`CustomerID` is only an identifier. For classic market segmentation, the model uses **Annual Income** and **Spending Score**, standardized so the features contribute on the same scale. Age and gender remain available for post-cluster profiling.

In [2]:
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Model matrix:", X_scaled.shape)

Model matrix: (200, 2)


## 4. Modeling
Compare K-Means and Agglomerative solutions for k=2…10 and inspect DBSCAN as a density-based alternative.

In [3]:
# K-Means model selection
k_results=[]
for k in range(2,11):
    model=KMeans(n_clusters=k, random_state=42, n_init=20)
    labels=model.fit_predict(X_scaled)
    k_results.append((k, model.inertia_, silhouette_score(X_scaled, labels)))
k_results=pd.DataFrame(k_results, columns=["k","inertia","silhouette"])

# Agglomerative comparison
a_results=[]
for k in range(2,11):
    labels=AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_scaled)
    a_results.append((k, silhouette_score(X_scaled, labels)))
a_results=pd.DataFrame(a_results, columns=["k","silhouette"])

print(k_results)
print(a_results)

 k    inertia  silhouette
 2 269.691012    0.321271
 3 157.704008    0.466585
 4 108.921317    0.493907
 5  65.568408    0.554657
 6  55.057348    0.539880
 7  44.864756    0.528149
 8  37.148117    0.456721
 9  32.392268    0.457085
10  29.685788    0.436212

 k  silhouette
 2    0.384234
 3    0.461048
 4    0.492551
 5    0.553809
 6    0.538676
 7    0.519795
 8    0.430862
 9    0.437690
10    0.433901


## 5. Evaluation
**Selected: K-Means with k=5**. It achieved silhouette **0.555** and produces clear, actionable income/spending groups. Agglomerative was competitive, while DBSCAN was less suitable for this compact business segmentation task.

In [4]:
final_model = KMeans(n_clusters=5, random_state=42, n_init=50)
df["Cluster"] = final_model.fit_predict(X_scaled)
print("Final silhouette:", silhouette_score(X_scaled, df["Cluster"]))

Final silhouette: 0.554657


### Segment Profiles

In [5]:
# Profile clusters with age, income and spending
profile = df.groupby("Cluster").agg(
    Customers=("CustomerID","count"),
    Avg_Age=("Age","mean"),
    Avg_Income_k=("Annual Income (k$)","mean"),
    Avg_Spending=("Spending Score (1-100)","mean")
).round(1)
display(profile)

 Cluster               Segment  Customers   Avg_Age  Avg_Income_k  Avg_Spending  Share_%
       0            Mainstream         81 42.716049     55.296296     49.518519     40.5
       1      Premium Spenders         39 32.692308     86.538462     82.128205     19.5
       2 Value-Driven Spenders         22 25.272727     25.727273     79.363636     11.0
       3  High-Income Cautious         35 41.114286     88.200000     17.114286     17.5
       4      Budget Conscious         23 45.217391     26.304348     20.913043     11.5


## 6. Deployment / Data Science Admin Dashboard
The dashboard summarizes customer count, selected segment count, model quality, and the most important business profiles.

In [6]:
from IPython.display import Image, display
display(Image(filename="images/admin_dashboard.png"))

![Admin Dashboard](images/admin_dashboard.png)

## Business Recommendations
- Protect premium/high-spending customers with loyalty and VIP experiences.
- Convert high-income cautious shoppers using personalized bundles and quality messaging.
- Retain value-driven spenders with frequent affordable offers.
- Use discount/value communication for budget-conscious shoppers.
- Use broad cross-sell campaigns for mainstream customers.

**Limitation:** these are behavioral clusters from a small learning dataset; they are not causal segments or permanent customer identities.